In [9]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pickle as pk
import os
from sklearn.model_selection import train_test_split
import streamlit as st



ModuleNotFoundError: No module named 'sklearn'

In [ ]:

cars_data = pd.read_csv("Cardetails.csv")

cars_data.drop(columns=['torque'], inplace=True)

print("Dataset Loaded... \nShape:", cars_data.shape)
cars_data.head()


Dataset Loaded... 
Shape: (8128, 12)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,5.0


In [ ]:

cars_data.dropna(inplace=True)
cars_data.drop_duplicates(inplace=True)

print("Cleaned Data... \nShape:", cars_data.shape)


Cleaned Data... 
Shape: (6718, 12)


In [ ]:

def get_brand_name(car_name):
    return car_name.split(' ')[0].strip()

def get_model_name(car_name):
    parts = car_name.split(' ', 1)
    return parts[1].strip() if len(parts) > 1 else ""

cars_data['brand'] = cars_data['name'].apply(get_brand_name)
cars_data['model'] = cars_data['name'].apply(get_model_name)

print("Brand & Model Extracted...")
cars_data[['brand', 'model']].head()


Brand & Model Extracted...


,brand,model
0,Maruti,Swift Dzire VDI
1,Skoda,Rapid 1.5 TDI Ambition
2,Honda,City 2017-2020 EXi
3,Hyundai,i20 Sportz Diesel
4,Maruti,Swift VXI BSIII


In [ ]:

unwanted_brands = ['Chevrolet', 'Datsun', 'Mitsubishi', 'Daewoo', 
                   'Fiat', 'Force', 'Ashok', 'Opel']

cars_data = cars_data[~cars_data['brand'].isin(unwanted_brands)].reset_index(drop=True)

print("Removed unwanted brands... \nShape:", cars_data.shape)


Removed unwanted brands... 
Shape: (6386, 14)


In [ ]:

def clean_data(value):
    value = value.split(' ')[0]
    value = value.strip()
    if value == '':
        value = 0
    return float(value)

cars_data['mileage'] = cars_data['mileage'].apply(clean_data)
cars_data['max_power'] = cars_data['max_power'].apply(clean_data)
cars_data['engine'] = cars_data['engine'].apply(clean_data)

print("Cleaned numeric columns...")


Cleaned numeric columns...


In [ ]:

owner_mapping = {
    'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3,
    'Fourth & Above Owner': 4, 'Test Drive Car': 5
}
fuel_mapping = {'Diesel': 1, 'Petrol': 2, 'LPG': 3, 'CNG': 4}
seller_type_mapping = {'Individual': 1, 'Dealer': 2, 'Trustmark Dealer': 3}
transmission_mapping = {'Manual': 1, 'Automatic': 2}
brand_mapping = {
    'Maruti': 1, 'Skoda': 2, 'Honda': 3, 'Hyundai': 4, 'Toyota': 5,
    'Ford': 6, 'Renault': 7, 'Mahindra': 8, 'Tata': 9, 'Jeep': 10,
    'Mercedes-Benz': 11, 'Audi': 12, 'Volkswagen': 13, 'BMW': 14,
    'Nissan': 15, 'Lexus': 16, 'Jaguar': 17, 'Land': 18, 'MG': 19,
    'Volvo': 20, 'Kia': 21, 'Ambassador': 22, 'Isuzu': 23
}

# Apply mappings
cars_data['owner'] = cars_data['owner'].map(owner_mapping)
cars_data['fuel'] = cars_data['fuel'].map(fuel_mapping)
cars_data['seller_type'] = cars_data['seller_type'].map(seller_type_mapping)
cars_data['transmission'] = cars_data['transmission'].map(transmission_mapping)
cars_data['brand'] = cars_data['brand'].map(brand_mapping)

print("Encoding Done...")


Encoding Done...


In [ ]:

feature_columns = ['brand', 'year', 'km_driven', 'fuel', 
                   'seller_type', 'transmission', 'owner', 
                   'mileage', 'engine', 'max_power', 'seats']

input_data = cars_data[feature_columns]
output_data = cars_data['selling_price']

x_train, x_test, y_train, y_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42
)

print("Data Split Done... \nTrain Size:", x_train.shape, "\nTest Size:", x_test.shape)
print("Training feature names:", x_train.columns.tolist())


Data Split Done... 
Train Size: (5108, 11) 
Test Size: (1278, 11)
Training feature names: ['brand', 'year', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'seats']


In [ ]:

model = LinearRegression()
model.fit(x_train, y_train)

print("Model Training Done...")


Model Training Done...


In [ ]:

y_pred = model.predict(x_test)

r2 = r2_score(y_test, y_pred)*100
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("📊 Model Evaluation:")
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")


📊 Model Evaluation:
R² Score: 57.3138
MAE: 182216.75
RMSE: 333273.24


In [ ]:

sample_input = pd.DataFrame(
    [[5, 2022, 12000, 1, 1, 1, 1, 12.99, 2494.0, 100.6, 5.0]],
    columns=feature_columns
)

print("Sample Input:", sample_input)
print("Sample Prediction:", model.predict(sample_input))


Sample Input:    brand  year  km_driven  fuel  seller_type  transmission  owner  mileage  \
0      5  2022      12000     1            1             1      1    12.99   

   engine  max_power  seats  
0  2494.0      100.6    5.0  
Sample Prediction: [1023580.9738985]


In [ ]:

file_path = os.path.join(os.getcwd(), "model.pkl")
pk.dump(model, open(file_path, "wb"))

print("✅ Model saved at:", file_path)


✅ Model saved at: c:\Users\OM MAKWANA\Documents\SEM5\AIML\COLLAGE\ML\model.pkl
